In [2]:
%load_ext autoreload

In [ ]:
import altair as alt

import geopandas as gpd
import polars as pl

%autoreload
from altair_utils import color_value_transit_teal
from downtown_travel_study_utils import parse_geog

In [4]:
def parse_telework_freq(persons, year):
    if year == 2018:
        replace_dict = {1: "5+", 2: "5+", 3: "4", 4: "2-3",
    5: "1", 6: "0-1 (<1)", 7: "0-1 (<1)", 8: "0 (never)", 995: "-1"}
    elif year == 2022:
        replace_dict = {1: "5+", 2: "5+", 3: "4", 4: "2-3",
    5: "2-3", 6: "1", 7: "0-1 (<1)", 8: "0-1 (<1)", 996: "0 (never)", 995: "-1"}
    else:
        raise NotImplementedError()
    return persons.with_columns(
        telework_freq_days_per_week=pl.col("telework_freq").replace_strict(replace_dict))


def parse_employment(persons, year):
    # parse to include/exclude self-employed (3)
    if year == 2022:
        replace_dict_inc_self = {1: True, 2: True, 3: True,
            5: False, 6: False, 7: False, 8: False, 995: None}
        replace_dict_exc_self = {1: True, 2: True, 3: False,
            5: False, 6: False, 7: False, 8: False, 995: None}
    elif year == 2018:
        replace_dict_inc_self = {1: True, 2: True, 3: True, 6: False, 7: False, 995: None}
        replace_dict_exc_self = {1: True, 2: True, 3: False, 6: False, 7: False, 995: None}
    else:
        raise NotImplementedError()
    return persons.with_columns(
        employed_inc_self=pl.col("employment").replace_strict(replace_dict_inc_self),
        employed_exc_self=pl.col("employment").replace_strict(replace_dict_exc_self),
    )


def county_to_county_telework_freq(persons):
    cols = ["home_county", "work_county", "telework_freq_days_per_week"]
    return persons.group_by(cols).agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort(cols)

In [19]:
person_2018 = (parse_telework_freq(pl.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2018\Processing_20211018\2_tour_extract\wt_wkday\survey2018_precx.dat", separator=" ").join(
    pl.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2018\Deliverable_20211018\person.csv", columns=["hh_id", "person_num", "telework_freq", "work_county_fips", "employment", "age"]),
    left_on=["hhno", "pno"], right_on=["hh_id", "person_num"]
).join(
        pl.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2018\Deliverable_20211018\hh.csv", columns=["hh_id", "home_county_fips"]),
        left_on="hhno", right_on="hh_id"
    )
    .filter(pl.col("age") > 3)  # filter out age < 18 y/o
    .select(
        "telework_freq", "psexpfac", "employment", home_county=pl.col("home_county_fips"), work_county=pl.col("work_county_fips")
    ),
    2018))
person_2022 = (parse_telework_freq(pl.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\03b-assign_day\wt-wkday_3day\person.csv").join(
    pl.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\01-taz_spatial_join\person.csv", columns=["hh_id", "person_num", "telework_freq", "work_county", "employment", "age"]),
    left_on=["hhno", "pno"], right_on=["hh_id", "person_num"]
    ).join(
        pl.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\01-taz_spatial_join\hh.csv", columns=["hh_id", "home_county"]),
        left_on="hhno", right_on="hh_id"
    )
    .filter(pl.col("age") > 3)  # filter out age < 18 y/o
    .select(
        "telework_freq", "psexpfac", "employment", home_county=(pl.col("home_county") % 100), work_county=(pl.col("work_county").cast(int) % 100),
    ),
    2022))

In [20]:
county_to_county_telework_freq_df = pl.concat(
    [
        county_to_county_telework_freq(person_2018).with_columns(year=2018),
        county_to_county_telework_freq(person_2022).with_columns(year=2022),
    ]
)
county_to_county_telework_freq_df.write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\county_to_county_telework_freq-adults_only-2018vs2022.csv")

In [29]:
person_2018.group_by("home_county", "work_county", "employment").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("home_county", "work_county", "employment").write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\employment-by_home_&_work_county-18+-2018.csv")
person_2022.group_by("home_county", "work_county", "employment").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("home_county", "work_county", "employment").write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\employment-by_home_&_work_county-18+-2022.csv")

In [21]:
employed_by_home_county = pl.concat([
    parse_employment(person_2018, 2018).group_by("home_county", "employed_exc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("home_county", "employed_exc_self").with_columns(year=2018),
    parse_employment(person_2022, 2022).group_by("home_county", "employed_exc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("home_county", "employed_exc_self").with_columns(year=2022)
])
employed_by_home_county.write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\employed-by_home_county-exc_self_employed-adults_only-2018vs2022.csv")

In [22]:
employed_by_work_county = pl.concat([
    parse_employment(person_2018, 2018).group_by("work_county", "employed_exc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("work_county", "employed_exc_self").with_columns(year=2018),
    parse_employment(person_2022, 2022).group_by("work_county", "employed_exc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("work_county", "employed_exc_self").with_columns(year=2022)
])
employed_by_work_county.write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\employed-by_work_county-exc_self_employed-adults_only-2018vs2022.csv")

In [23]:
# by home county, inc self employed
pl.concat([
    parse_employment(person_2018, 2018).group_by("home_county", "employed_inc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("home_county", "employed_inc_self").with_columns(year=2018),
    parse_employment(person_2022, 2022).group_by("home_county", "employed_inc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("home_county", "employed_inc_self").with_columns(year=2022)
]).write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\employed-by_home_county-inc_self_employed-adults_only-2018vs2022.csv")

In [24]:
# by work county, inc self employed
pl.concat([
    parse_employment(person_2018, 2018).group_by("work_county", "employed_inc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("work_county", "employed_inc_self").with_columns(year=2018),
    parse_employment(person_2022, 2022).group_by("work_county", "employed_inc_self").agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort("work_county", "employed_inc_self").with_columns(year=2022)
]).write_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241115\summaries\employed-by_work_county-inc_self_employed-adults_only-2018vs2022.csv")